In [1]:
# Mount Google Drive if needed
from google.colab import drive
drive.mount('/content/drive')

# Download the Cornell Movie Dialogs corpus
!wget http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
!unzip cornell_movie_dialogs_corpus.zip


Mounted at /content/drive
--2025-06-12 19:23:04--  http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
Resolving www.cs.cornell.edu (www.cs.cornell.edu)... 132.236.207.53
Connecting to www.cs.cornell.edu (www.cs.cornell.edu)|132.236.207.53|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip [following]
--2025-06-12 19:23:05--  https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
Connecting to www.cs.cornell.edu (www.cs.cornell.edu)|132.236.207.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9916637 (9.5M) [application/zip]
Saving to: ‘cornell_movie_dialogs_corpus.zip’

cornell_movie_dialo 100%[===================>]   9.46M  12.5MB/s    in 0.8s    

2025-06-12 19:23:06 (12.5 MB/s) - ‘cornell_movie_dialogs_corpus.zip’ saved [9916637/9916637]

Archive:  cornell_movie_dialogs_corpus.zip
   creating: cornell

In [2]:
import os
import re

lines_path = "cornell movie-dialogs corpus/movie_lines.txt"
conversations_path = "cornell movie-dialogs corpus/movie_conversations.txt"

# Load movie lines
id2line = {}
with open(lines_path, encoding='iso-8859-1') as file:
    for line in file:
        parts = line.strip().split(" +++$+++ ")
        if len(parts) == 5:
            id2line[parts[0]] = parts[4]

# Load conversations
conversations = []
with open(conversations_path, encoding='iso-8859-1') as file:
    for line in file:
        parts = line.strip().split(" +++$+++ ")
        utterance_ids = eval(parts[-1])  # convert string list to list
        conversation = [id2line[uid] for uid in utterance_ids if uid in id2line]
        conversations.append(conversation)

# Extract question-answer pairs
pairs = []
for conv in conversations:
    for i in range(len(conv) - 1):
        input_line = conv[i].strip()
        target_line = conv[i + 1].strip()
        if input_line and target_line:
            pairs.append((input_line, target_line))


In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Download punkt_tab
from nltk.tokenize import word_tokenize

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9?.!,']+", " ", text)
    text = re.sub(r"['’]", "'", text)
    return text

def tokenize_pair(pair):
    return (
        word_tokenize(clean_text(pair[0])),
        word_tokenize(clean_text(pair[1]))
    )

tokenized_pairs = [tokenize_pair(pair) for pair in pairs]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
from collections import Counter

# Flatten and count all words
all_words = [word for pair in tokenized_pairs for sentence in pair for word in sentence]
word_counts = Counter(all_words)

# Set vocabulary size limit
vocab_size = 10000
most_common = word_counts.most_common(vocab_size - 4)  # reserve 4 tokens

# Special tokens
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

# Create vocab dicts
vocab = [PAD, SOS, EOS, UNK] + [word for word, _ in most_common]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

# Encode with special tokens
def encode(sentence, word2idx):
    return [word2idx.get(word, word2idx[UNK]) for word in sentence]

encoded_pairs = []
for q, a in tokenized_pairs:
    q_encoded = encode(q, word2idx)
    a_encoded = [word2idx[SOS]] + encode(a, word2idx) + [word2idx[EOS]]
    encoded_pairs.append((q_encoded, a_encoded))


In [6]:
import random

# Shuffle and split
random.shuffle(encoded_pairs)
split_index = int(0.9 * len(encoded_pairs))
train_data = encoded_pairs[:split_index]
val_data = encoded_pairs[split_index:]

print(f"Total Pairs: {len(encoded_pairs)}")
print(f"Training: {len(train_data)}, Validation: {len(val_data)}")


Total Pairs: 221367
Training: 199230, Validation: 22137


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
import pickle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [9]:
import pickle

# Save the data
with open("train_data.pkl", "wb") as f:
    pickle.dump(train_data, f)

with open("val_data.pkl", "wb") as f:
    pickle.dump(val_data, f)

vocab_data = {"word2idx": word2idx, "idx2word": idx2word}
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab_data, f)

print("Data and vocabulary saved to pickle files.")

Data and vocabulary saved to pickle files.


In [10]:
with open("train_data.pkl", "rb") as f:
    train_data = pickle.load(f)

with open("val_data.pkl", "rb") as f:
    val_data = pickle.load(f)

with open("vocab.pkl", "rb") as f:
    vocab_data = pickle.load(f)
    word2idx = vocab_data["word2idx"]
    idx2word = vocab_data["idx2word"]

vocab_size = len(word2idx)
PAD_IDX = word2idx["<PAD>"]
SOS_IDX = word2idx["<SOS>"]
EOS_IDX = word2idx["<EOS>"]

In [11]:
class ChatDataset(Dataset):
    def __init__(self, data, max_len=20):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def pad(self, seq):
        seq = seq[:self.max_len]
        return seq + [PAD_IDX] * (self.max_len - len(seq))

    def __getitem__(self, idx):
        q, a = self.data[idx]
        return torch.tensor(self.pad(q)), torch.tensor(self.pad(a))

batch_size = 64
train_loader = DataLoader(ChatDataset(train_data), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(ChatDataset(val_data), batch_size=batch_size)


In [12]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_IDX)
        self.rnn = nn.GRU(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)
        outputs, hidden = self.rnn(emb)
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, 1)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.size(1)
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = self.attn(torch.cat((hidden, encoder_outputs), dim=2)).squeeze(2)
        attn_weights = torch.softmax(energy, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_IDX)
        self.rnn = nn.GRU(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.attention = Attention(hidden_size)

    def forward(self, x, hidden, encoder_outputs):
        x = x.unsqueeze(1)
        emb = self.embedding(x)
        context = self.attention(hidden[-1], encoder_outputs).unsqueeze(1)
        rnn_input = torch.cat((emb, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        out_vocab = self.fc(output.squeeze(1))
        return out_vocab, hidden


In [13]:
embed_size = 256
hidden_size = 512
num_epochs = 10
learning_rate = 0.001

encoder = Encoder(vocab_size, embed_size, hidden_size).to(device)
decoder = Decoder(vocab_size, embed_size, hidden_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)

def train_step(input_batch, target_batch):
    encoder.train()
    decoder.train()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_batch, target_batch = input_batch.to(device), target_batch.to(device)

    encoder_outputs, encoder_hidden = encoder(input_batch)

    decoder_input = torch.full((input_batch.size(0),), SOS_IDX, dtype=torch.long, device=device)
    decoder_hidden = encoder_hidden
    loss = 0

    for t in range(target_batch.size(1)):
        decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_outputs)
        loss += criterion(decoder_output, target_batch[:, t])
        decoder_input = target_batch[:, t]  # teacher forcing

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_batch.size(1)


In [14]:
for epoch in range(num_epochs):
    total_loss = 0
    for i, (inputs, targets) in enumerate(train_loader):
        loss = train_step(inputs, targets)
        total_loss += loss

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")


Epoch 1/10, Loss: 4.0379
Epoch 2/10, Loss: 3.6038
Epoch 3/10, Loss: 3.3510
Epoch 4/10, Loss: 3.1212
Epoch 5/10, Loss: 2.9249
Epoch 6/10, Loss: 2.7683
Epoch 7/10, Loss: 2.6421
Epoch 8/10, Loss: 2.5429
Epoch 9/10, Loss: 2.4653
Epoch 10/10, Loss: 2.4038
